# Auto Tagging Support Tickets Using Large Language Models (LLMs)

## Objective

The objective of this project is to automatically categorize customer support tickets using a Large Language Model (LLM). Support teams receive a large number of user queries every day, and manually tagging these tickets into categories can be time-consuming. 

This project aims to build an automated system that reads a support ticket and predicts the **top three most relevant categories** for that ticket.

## Dataset

The **Banking77 dataset** is used in this project. It contains customer service queries related to banking applications such as payments, card issues, account access, and transactions. In this project, each query is treated as a support ticket.

## Approach

The system uses a pre-trained transformer model for **zero-shot classification**, which allows the model to categorize text without being specifically trained on the dataset.

Two approaches are implemented:

1. **Zero-Shot Classification**  
   The model predicts the most relevant ticket categories using only predefined labels without any examples.

2. **Few-Shot Learning**  
   The model is provided with a few example tickets and their corresponding tags before classifying a new ticket. This helps the model better understand the task and may improve prediction quality.

The model outputs the **top three predicted tags** for each support ticket, allowing support systems to route issues more efficiently.

In [1]:
import pandas as pd
import numpy as np
import torch

from datasets import load_dataset
from transformers import pipeline
from tqdm import tqdm

c:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Dataset

In [2]:
dataset = load_dataset("banking77")

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})

# Convert Dataset to DataFrame

To make the dataset easier to manipulate and visualize,
we convert it into a pandas DataFrame.



In [3]:
df = pd.DataFrame(dataset["train"])

df.head()

,text,label
0,I am still waiting on my card?,11
1,What can I do if my card still hasn't arrived ...,11
2,I have been waiting over a week. Is the card s...,11
3,Can I track my card while it is in the process...,11
4,"How do I know if I will get my card, or if it ...",11


# Extract Ticket Text

The dataset contains two columns:

- text → customer query
- label → category id

For my project, the "text" column will represent the
support ticket submitted by the customer.

In [4]:
df = df[["text"]]

df = df.rename(columns={"text": "ticket"})

df.head()

,ticket
0,I am still waiting on my card?
1,What can I do if my card still hasn't arrived ...
2,I have been waiting over a week. Is the card s...
3,Can I track my card while it is in the process...
4,"How do I know if I will get my card, or if it ..."


In [5]:
df.shape

(10003, 1)

# Reduce Dataset Size

To make the project run faster, we select a small subset
of the dataset.



In [6]:
df = df.sample(100, random_state=42)

df.head()

,ticket
6883,Is it possible for me to change my PIN number?
5836,I'm not sure why my card didn't work
8601,I don't think my top up worked
2545,Can you explain why my payment was charged a fee?
8697,How long does a transfer from a UK account tak...


## Define Support Ticket Tags
Before classification, we define possible support
ticket categories.

In [8]:
labels = [
    "Account Access Problem",
    "Payment or Billing Issue",
    "Card Problem",
    "Transaction Issue",
    "Technical App Problem",
    "Refund Request",
    "Fraud or Security Issue",
    "General Inquiry"
]

labels

['Account Access Problem',
 'Payment or Billing Issue',
 'Card Problem',
 'Transaction Issue',
 'Technical App Problem',
 'Refund Request',
 'Fraud or Security Issue',
 'General Inquiry']

# Load Zero-Shot Classification Model

We use a pre-trained Large Language Model from HuggingFace
for zero-shot classification.


Zero-shot learning allows the model to classify text into categories
without being trained on this specific dataset.

The model understands language patterns and predicts which
category best matches the input text.

In [9]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="./distilbert-mnli"
)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 37196.86it/s]


# Test Zero-Shot Classification

 - Test the model on a single support ticket to ensure
the classification pipeline works correctly.

-The model will compare the ticket text with the predefined labels
and return probabilities for each label.

In [10]:
ticket = df["ticket"].iloc[0]

result = classifier(ticket, labels)

result

{'sequence': 'Is it possible for me to change my PIN number?',
 'labels': ['Transaction Issue',
  'Payment or Billing Issue',
  'Card Problem',
  'General Inquiry',
  'Account Access Problem',
  'Fraud or Security Issue',
  'Technical App Problem',
  'Refund Request'],
 'scores': [0.21937815845012665,
  0.20915454626083374,
  0.14060351252555847,
  0.11425542086362839,
  0.1119743138551712,
  0.07972832024097443,
  0.07058710604906082,
  0.0543186292052269]}

# Extract Top 3 Tags

- The model returns probabilities for all labels.
We select the top three labels with the highest scores.

- Ranking predictions allows the system to suggest multiple
possible categories for a support ticket.

In [11]:
top3 = result["labels"][:3]

top3

['Transaction Issue', 'Payment or Billing Issue', 'Card Problem']

# Apply Zero-Shot Classification to All Tickets

- Now run the model on multiple support tickets and
store the top three predicted tags.

- This step automates the tagging of customer support tickets
using an LLM.

In [12]:
zero_shot_predictions = []

for ticket in df["ticket"]:
    
    result = classifier(ticket, labels)
    
    top3 = result["labels"][:3]
    
    zero_shot_predictions.append(top3)

df["zero_shot_tags"] = zero_shot_predictions

df.head()

,ticket,zero_shot_tags
6883,Is it possible for me to change my PIN number?,"[Transaction Issue, Payment or Billing Issue, ..."
5836,I'm not sure why my card didn't work,"[Card Problem, Transaction Issue, Refund Request]"
8601,I don't think my top up worked,"[General Inquiry, Transaction Issue, Payment o..."
2545,Can you explain why my payment was charged a fee?,"[Transaction Issue, Refund Request, Account Ac..."
8697,How long does a transfer from a UK account tak...,"[Transaction Issue, General Inquiry, Account A..."


# Few-Shot Learning for Ticket Tagging

- In few-shot learning we provide a few labeled examples before
classifying new tickets.

- Few-shot learning helps the model understand the task better
by showing some examples of tickets and their correct tags.

This often improves classification accuracy compared to
zero-shot learning.

In [13]:
examples = """
Ticket: I forgot my password and cannot login
Tag: Account Access Problem

Ticket: My card payment failed
Tag: Card Problem

Ticket: I was charged twice for my order
Tag: Payment or Billing Issue
"""

# Create Few-Shot Prompt

- Build a prompt that contains example tickets along with
their correct tags, followed by the new ticket that needs
to be classified.

- The model learns the pattern from the examples and applies
it to the new ticket.

In [14]:
def few_shot_prompt(ticket):

    prompt = f"""
    You are a support ticket classifier.

    Possible categories:
    {labels}

    Here are some examples:

    {examples}

    Now classify the following support ticket and return the
    three most relevant categories.

    Ticket: {ticket}
    """

    return prompt

# Test Few-Shot Prompt

- Test the prompt with one ticket before applying it to
the full dataset.

- Testing with a single example ensures that the prompt is
structured correctly.

In [15]:
ticket = df["ticket"].iloc[0]

prompt = few_shot_prompt(ticket)

print(prompt)


    You are a support ticket classifier.

    Possible categories:
    ['Account Access Problem', 'Payment or Billing Issue', 'Card Problem', 'Transaction Issue', 'Technical App Problem', 'Refund Request', 'Fraud or Security Issue', 'General Inquiry']

    Here are some examples:

    
Ticket: I forgot my password and cannot login
Tag: Account Access Problem

Ticket: My card payment failed
Tag: Card Problem

Ticket: I was charged twice for my order
Tag: Payment or Billing Issue


    Now classify the following support ticket and return the
    three most relevant categories.

    Ticket: Is it possible for me to change my PIN number?
    


## Few-Shot Prompt for All Tickets

In [16]:
few_shot_predictions = []

for ticket in df["ticket"]:

    prompt = few_shot_prompt(ticket)

    result = classifier(prompt, labels)

    top3 = result["labels"][:3]

    few_shot_predictions.append(top3)

df["few_shot_tags"] = few_shot_predictions

df.head()

,ticket,zero_shot_tags,few_shot_tags
6883,Is it possible for me to change my PIN number?,"[Transaction Issue, Payment or Billing Issue, ...","[Card Problem, Transaction Issue, Payment or B..."
5836,I'm not sure why my card didn't work,"[Card Problem, Transaction Issue, Refund Request]","[Fraud or Security Issue, Transaction Issue, P..."
8601,I don't think my top up worked,"[General Inquiry, Transaction Issue, Payment o...","[Payment or Billing Issue, Fraud or Security I..."
2545,Can you explain why my payment was charged a fee?,"[Transaction Issue, Refund Request, Account Ac...","[Payment or Billing Issue, Fraud or Security I..."
8697,How long does a transfer from a UK account tak...,"[Transaction Issue, General Inquiry, Account A...","[Payment or Billing Issue, Fraud or Security I..."


# Compare Zero-Shot and Few-Shot Results

- We compare the predicted tags generated by zero-shot
and few-shot approaches.


- This comparison helps evaluate whether providing
examples improves classification quality.

In [17]:
comparison_df = df[["ticket", "zero_shot_tags", "few_shot_tags"]]

comparison_df.head(10)

,ticket,zero_shot_tags,few_shot_tags
6883,Is it possible for me to change my PIN number?,"[Transaction Issue, Payment or Billing Issue, ...","[Card Problem, Transaction Issue, Payment or B..."
5836,I'm not sure why my card didn't work,"[Card Problem, Transaction Issue, Refund Request]","[Fraud or Security Issue, Transaction Issue, P..."
8601,I don't think my top up worked,"[General Inquiry, Transaction Issue, Payment o...","[Payment or Billing Issue, Fraud or Security I..."
2545,Can you explain why my payment was charged a fee?,"[Transaction Issue, Refund Request, Account Ac...","[Payment or Billing Issue, Fraud or Security I..."
8697,How long does a transfer from a UK account tak...,"[Transaction Issue, General Inquiry, Account A...","[Payment or Billing Issue, Fraud or Security I..."
5573,Why am I getting declines when trying to make ...,"[Transaction Issue, General Inquiry, Account A...","[Card Problem, Transaction Issue, Account Acce..."
576,What is the $1 transaction on my account?,"[Transaction Issue, Account Access Problem, Fr...","[Payment or Billing Issue, Card Problem, Fraud..."
6832,It looks like my card payment was sent back.,"[Card Problem, Transaction Issue, Refund Request]","[Card Problem, Payment or Billing Issue, Trans..."
7111,Why am I unable to transfer money when I was a...,"[Transaction Issue, Account Access Problem, Re...","[Payment or Billing Issue, Card Problem, Fraud..."
439,What if there is an error on the exchange rate?,"[Transaction Issue, Payment or Billing Issue, ...","[Card Problem, Transaction Issue, Payment or B..."


# Conclusion

In this project, I built an automatic support ticket
tagging system using a Large Language Model.

The Banking77 dataset was used as a collection of
customer queries representing support tickets.

Two approaches were explored:

• Zero-shot classification
• Few-shot classification

The system successfully generated the top three
predicted categories for each ticket.

Few-shot prompting provided contextual examples
that helped improve the model's understanding
of the classification task.

This approach demonstrates how LLMs can automate
support ticket categorization and assist customer
service systems in efficiently routing issues.